# Kovai Finserv RAG — live walkthrough

Run the cells in order. The order is the point:

1. **Retrieval alone** — no model involved. If the right chunk is not here, no prompt will save the answer.
2. **The full turn** — retrieval + Claude, with sources and cost.
3. **Try to break it** — the refusal paths.

Prerequisite: `python scripts/ingest.py` has been run, so `chroma_db/` exists.

## Setup

`settings.chroma_path` is the relative path `chroma_db`, so the working directory has to be the project root. The guard below fixes it if you moved this notebook.

In [ ]:
import os
from pathlib import Path

# Walk up until we find the project root, so relative paths resolve.
while not Path("scripts/ingest.py").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
print("cwd            :", Path.cwd())
print("index present  :", Path("chroma_db").exists())

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import logging
logging.basicConfig(level=logging.INFO)

from app.rag import retrieve, answer_question

# Claude Haiku 4.5 pricing: $1 per 1M input tokens, $5 per 1M output tokens.
def cost(r):
    return round(r.input_tokens / 1e6 * 1 + r.output_tokens / 1e6 * 5, 6)

## STEP 1 — retrieval alone, before any model is involved

Always debug RAG in this order. If the right chunk is not here, no amount of prompt engineering will save the answer.

Expect `1. Foreclosure and Prepayment` at rank 1. The absolute scores are low (~0.47 for a direct hit) because `all-MiniLM-L6-v2` compresses short-question-vs-long-section similarity — read the ranking, not the magnitude.

In [ ]:
for h in retrieve("How much do I pay if I close my loan early?"):
    print(round(h["relevance"], 3), "|", h["title"])

## STEP 2 — the full turn

Watch for two things in the answer: the exact figure (**2% of the outstanding principal plus GST**, after 12 EMIs) and the **Kovai Shakti exception**. Rule 4 of the system prompt makes stating the exception mandatory — an answer that gives only the 2% is a failure even though it is not wrong.

In [ ]:
r = answer_question("How much do I pay if I close my loan early?")

print(r.answer)
print("\nSources:", r.sources)
print("Cost   : $", cost(r))

## STEP 3 — now try to break it

Three different refusal mechanisms, which is why all three are here:

| Question | Mechanism |
|---|---|
| Capital of France | **Retrieval floor.** Nothing clears `min_relevance`, so `retrieved=False` and the API is never called — zero tokens, zero cost. |
| CIBIL 600 | **Rule 7.** The Eligibility section *is* retrieved and does say 680 minimum, so this should quote the threshold without ruling on this specific customer. |
| "Foreclosure is free" | **Rule 3 vs. a false premise.** The friend is wrong except under Kovai Shakti after 18 EMIs. A good answer corrects the premise and states the exception. |

Only the first is a hard refusal. The other two are the interesting cases: the bot has the right document in hand and still has to not overclaim.

In [ ]:
for q in ["What is the capital of France?",
          "Can I get a loan if my CIBIL score is 600?",
          "My friend says foreclosure is free at Kovai. Is that right?"]:
    r = answer_question(q)
    print("Q:", q)
    print("A:", r.answer)
    print("   retrieved:", r.retrieved, "| sources:", r.sources, "| $", cost(r), "\n")

## The gap worth showing too

The handbook deliberately never mentions home loans. This is the case the retrieval floor does **not** catch — "home loan" is topically close to every sentence in a lending document, so four personal-loan sections clear the threshold at 0.30–0.40 and get handed to the model.

That makes it a test of rules 1 and 2, not of the distance threshold. Run it and see whether the model refuses on content it can read but that never answers the question.

In [ ]:
r = answer_question("Do you offer home loans?")
print(r.answer)
print("\nretrieved:", r.retrieved, "| sources:", r.sources, "| $", cost(r))